# <span style="color:red; font-size:30px"> Lecture 18b: Cleaning Data </span>

## <span style="color:red"> I. Import Libraries and Read in Files</span>

In [ ]:
import numpy as np
import pandas as pd

circuits = pd.read_csv("f1_data/circuits.csv")

# <span style="color:red"> II. Examine Data </span>

In [ ]:
# column/variable headings
print(circuits.columns.values)
print()
# number of rows and columns of DataFrame
print(circuits.shape)
print()
n_rows, n_cols = circuits.shape
print("# of rows:", n_rows)
print("# of columns:", n_cols)


<font size = "4">

We can also get the **DataTypes** for each column (dtypes)

In [ ]:
print(circuits.dtypes)

<font size = "4">

We've seen **attributes** (a.k.a properties) of Pandas DataFrames like ".shape", ".dtypes", and ".columns"

We've also seen **methods** like ".query", ".apply", and ".mean"


How do we remember all of these?

VS code can remind you of all available methods for an object.

If you want to see attributes as well, you should do the following.

1. Install the Pylance extension in your VS Code

2. Click the box at the center-top of your VS Code window

3. Start typing "> Developer: Reload Window". It will autocomplete after you type a sufficient number of the characters, then hit enter.

4. This will also restart the notebook, so you should re-run the cells above

In the code cell below, type "circuits." (with the period) and then wait, without running the cell.

In [ ]:
# Type the following (including the period), then wait, without running the cell
# type this (with the period) >> circuits.



<font size = "4">

**Note:** We've already seen how to add new columns/variables to a DataFrame. But we haven't seen how to rename the existing columns/variables. Some of the most straightforward are shown below

In [ ]:
# Change the first and last column headings
print(circuits.columns.values)
print()

# Change last heading to "Uniform Resource Locator"
circuits.columns.values[-1] = "Uniform Resource Locator"

# Change first heading to "ID_No"
circuits.columns.values[0] = "ID_No"
print(circuits.columns.values)

In [ ]:
# Change back to "url" 

# Create a new data-frame and assign it to "circuits". 
# So we're overwriting the dataframe.
circuits = circuits.rename(columns={'Uniform Resource Locator': 'url'})

print(circuits.columns.values)

In [ ]:
# Change back to "circuitID"

# Instead of making a new dataframe and over-writing the old one, change it "in-place"
circuits.rename(columns={'ID_No':'circuitID'}, inplace=True)

print(circuits.columns.values)

# <span style="color:red"> III. Cleaning Data </span>

<font size = "4">

Data that should be numerical (e.g. altitude of a Formula One track) can have missing values or non-numerical replacements (like "N/A"). We want to manipulate the data to allow for numerical calculations.

In [ ]:
print(circuits["lat"].mean())
print(circuits["lng"].mean())
print(circuits["alt"].mean())

In [ ]:
# Let's try to figure out the problem
print(circuits.dtypes)

# The column "alt" has the DataType "str" or "object" depending on your python version

In [ ]:
print(circuits["alt"])
# these look like numbers, except the last two are "\N"

In [ ]:
display(circuits["alt"])

<font size = "4">

- Each column of a Pandas DataFrame is a Pandas Series
- Each Pandas Series with "dtype: object" has a collection of methods known as "String Methods"
- To execute a method from this collection you must type: ``series_name.str.method_name``
- **NOTE**: Whenever I type something with a vague placeholder name like "series_name" or "method_name" this means that you should replace that word/phrase with an actual Python object or method/function name
- Within this collection, there is a method we want to use called ``.isnumeric``

In [ ]:
# This is a Pandas Series
print(type(circuits["alt"])) 
print()
# This is a Pandas StringMethods object (collection of methods/functions)
print(type(circuits["alt"].str)) 
print()
# This is a method/function belonging to the StringMethods collection
print(type(circuits["alt"].str.isnumeric)) 

<font size = "4">

You'll get an error if you try to use ``.str`` with a column that doesn't have the "object" dtype

In [ ]:
circuits["lat"].str

<font size = "4">

The ``series_name.str.isnumeric()`` method returns a Pandas Series with ``True`` everywhere the Series entry is deemed "numeric" and ``False`` otherwise.

In [ ]:
print(circuits["alt"].str.isnumeric())

In [ ]:
# We can reference subattributes of columns in ".query()"
non_numeric = circuits.query("  alt.str.isnumeric() == False ")
print(non_numeric)

<font size = "4">

Why is the string "-7" considered non-numeric?

In [ ]:
# The pd.unique() function extracts unique values from a series/array

unique_non_numeric = pd.unique( non_numeric["alt"] )
print(unique_non_numeric)

<font size = "4">

We'll decide what we should replace "\N" and "-7" with

- "-7" is easy. We'll replace the string "-7" with the integer -7
- What about "\N"? It turns out the best choice is the "nan" (not a number) object from the Numpy library.

In [ ]:
print(np.nan)

# ironically, the type of this object is a floating-point number...
print(type(np.nan))

In [ ]:
# unique_non_numeric is the list with ["-7", "\\N"]
# we'll make a list of the same length with the values we want to replace them with

replace_vals = [-7, np.nan]

# Overwrite the "alt" column, replacing every appearance from unique_non_numeric with
# the corresponding element of replace_vals
circuits["alt"] = circuits["alt"].replace(unique_non_numeric, replace_vals)

<font size = "4">

Did it work? Let's test it

In [ ]:
non_numeric = circuits.query("  alt.str.isnumeric() == False  ")
print(non_numeric)

<font size = "4">

There are no rows with a non-numeric value for "alt"! Are we done? (No)

In [ ]:
print(circuits["alt"].mean()) # get an error

<font size = "4">

What happened? We still don't have the right "dtype" to do calculations. We should have checked that in the first place

In [ ]:
print(circuits.dtypes)

<font size = "4">

The "alt" column still has the "object" dtype. But since we've cleaned the data, all of its elements **can be** converted to numeric values. The ``Pandas`` function ``to_numeric`` can be used to do this.

We'll create a new column to the DataFrame.

In [ ]:
circuits["alt_numeric"] = pd.to_numeric(circuits["alt"])
print(circuits["alt_numeric"].mean())

In [ ]:
# After the cleaning process is done, you may want to store the dataset again
# It's recommended to do this in a separate file from the original
# That way you can always go back to the original if you made a programming error

circuits.to_csv("f1_clean_data/circuits.csv")

<font size = "4">

- By the way, for this example, we can clean the altitude data with one line.

- We'll reload the data and show how.

In [ ]:
circuits_2 = pd.read_csv("f1_data/circuits.csv")

circuits_2["alt_numeric"] = pd.to_numeric(circuits_2["alt"], errors="coerce")

print(circuits_2["alt_numeric"].mean())

<font size = "4">

**Warning:** We cannot just "force-clean" every dataset like this. See the last exercise for an example where you CAN'T do this.

<font size = "4">

**Exercise**: When I was creating this notebook, I had the following question. "If I read in the .csv file I just created and save it as a new DataFrame, will I have to re-do any part of the cleaning process?" Test this out for yourself.

**Exercise**: Read in the .csv file that was just created, saving it as a new DataFrame. You'll see there is an extra column/variable called "Unnamed" with the values 0, 1, 2, ..., 75, 76. Type ``help(circuits.to_csv)`` to figure out how to create a new .csv file without this extra column.

In [ ]:
# your answer here (save the new cleaned csv to f1_clean_data/circuits_final.csv for use in lecture 18c)

In [ ]:
# your answer here


<font size = "4">

**Exercise:**

- Use ".replace()" with the "country" column
- Replace "USA" with "United States"

In [ ]:
# Write your own code





<font size = "4">

**Exercise:** 

Below, I manually create a DataFrame using made-up survey data. Each respondent is a student, and their answers were read in using Python code as strings.

- Clean the "age" column so that we can calculate the mean age of the respondents. Will using `pd.to_numeric` with `errors="coerce"` clean the data correctly?

<br>

<font size = "3">

Note: The line with curly braces ``{}`` creates a *dictionary* object, which is then converted to a Pandas DataFrame. We also saw this earlier in the notebook within the `.replace` method. We'll discuss dictionaries more in the next lecture.

In [ ]:
list_ages = ["twenty-one", "21", "18", "twenty", "", "21", "twenty five", "18", "twenty", "19", 
    "20", "nineteen", "", "17"]
list_major = ["DSci", "Math", "undeclared", "Undeclared", "Sociology", "psychology", "Spanish", 
    "Ancient Mediterranean Studies", "Bio", "biology", "Bus Admin.", "BBA", "Linguistics", ""]
list_graduate = ["Y", "Y", "N", "Y", "N", "N", "N", "N", "Y", "N", "N", "N", "Y", "N"]

data_dictionary = {"age": list_ages, "major": list_major, "graduating?": list_graduate}

df_survey = pd.DataFrame(data_dictionary)

display(df_survey)

# <span style="color:red"> IV. Practice: Cleaning the 2020 Election Data </span>

<font size = "4">

The folder `election_data` contains results from the 2020 U.S. general election. In these exercises, you'll look for problems in the data and fix them using the tools from this lecture.

Run the cell below to read in the files we'll use.

In [ ]:
pres_state = pd.read_csv("election_data/president_state.csv")
gov_state = pd.read_csv("election_data/governors_state.csv")
pres_county = pd.read_csv("election_data/president_county.csv")
pres_cand = pd.read_csv("election_data/president_county_candidate.csv")

<font size = "4">

**Exercise 1:** Examine `pres_state`

- Print the column names, the number of rows and columns, and the dtypes.

- Compute the mean of `total_votes`. Print the mean of `total_votes`.

- There are 50 states plus the District of Columbia, but `pres_state` has 52 rows. Use pd.unique() to find the value of state that is not one of the 50 states or the District of Columbia.

- Use `.query()` to create a new DataFrame called `pres_state_clean` without that row, and recompute the mean.

In [ ]:
# your answer here

<font size = "4">

**Exercise 2:** Consistent column names

- In `pres_state`, the vote column is called `total_votes`, but in `gov_state` it is called `votes`.

- Use `.rename()` to change the name of the `votes` column of `gov_state` to `total_votes`. Print the column names to check your work.

In [ ]:
# your answer here

<font size = "4">

**Exercise 3:** Hidden characters

- Use `.query()` to find all rows of `pres_cand` where `candidate` is equal to `"Write-ins"`. How many rows do you get?

- Now use `pd.unique()` to print all of the unique values of the `candidate` column. Find the exact string used for write-in votes in pd.unique(), including any spaces, and use it in a .query() that returns the write-in rows.

- Use `.replace()` to fix every candidate name with this problem (there are two of them), then re-run your query from the first step.

In [ ]:
# your answer here

In [ ]:
# your answer here

<font size = "4">

**Exercise 4:** Don't assume what a code means

- Use `pd.unique()` to look at the values of the `party` column of `pres_cand`. Most of them are three-letter abbreviations.

- Many people read "GOP" as the Republican party. Use `.query()` to see which candidate(s) are listed with party `"GOP"`, and compare with the candidate(s) listed with party `"REP"`.

- Create a new column `party_name` that is a copy of `party` where `"DEM"` is replaced with `"Democratic"`, `"REP"` with `"Republican"`, `"LIB"` with `"Libertarian"`, and `"GRN"` with `"Green"`. Leave every other code, including `"GOP"`, unchanged.

- Use `.query()` to select the rows where `party_name` is `"Republican"`, and print the unique values of `candidate` in those rows. Only one name should appear.

In [ ]:
# your answer here

In [ ]:
# your answer here

<font size = "4">

**Exercise 5:** Cleaning numbers that were typed by hand

Below, I manually create a DataFrame using *made-up* data: a volunteer typed up the number of presidential votes counted in eight Georgia counties, and the entries were read in as strings.

- Find the entries of `votes_typed` that are not numeric, and decide what each one should be replaced with.

- Create a column `votes_clean` that contains numbers, and compute its mean.

- Create a second column votes_coerce using `pd.to_numeric` with `errors="coerce"`, and print the means of votes_clean and votes_coerce.

- **Check your work:** merge `df_typed` with the `state`, `county`, and `current_votes` columns of `pres_county`, using `on = ["state", "county"]` (as in Lecture 18a, Exercise 6). Compare `votes_clean` with `current_votes`. Use .query() to display the row where votes_clean is missing, including its current_votes value.

In [ ]:
list_counties = ["Appling County", "Atkinson County", "Bacon County", "Baker County",
    "Baldwin County", "Banks County", "Barrow County", "Bartow County"]
list_votes_typed = ["8,390", "3155", "4667", "N/A", "18251", "8,801", "37921", "50,464"]

df_typed = pd.DataFrame({"state": "Georgia", "county": list_counties, "votes_typed": list_votes_typed})

display(df_typed)

In [ ]:
# your answer here

In [ ]:
# your answer here

<font size = "4">

**Exercise 6:** Save your work

- Save `pres_state_clean` (from Exercise 1) to a new file called `election_data/president_state_clean.csv`, without the extra index column.

- Read the file back in and check that it has 51 rows, 2 columns, and no "Unnamed" column.

In [ ]:
# your answer here